# 2.3 章节实践

本节只验收 910B3 SIMD/AIV + RTC 主线；950 SIMT 始终保持待验证，不能要求学生伪造结果。

## 一、客观题

1. pair-planar `[384]` 如何对应 interleaved `[12,64]`？
2. `AIV_ONLY` 表示运行在哪类 Core？
3. SIMD 与 SIMT 的并行单位分别是什么？
4. position=0 时，cos、sin 和输出分别应是什么？
5. 判断：`rope_simt_950.asc` 文件存在即可写 SIMT PASS。


## 二、简单编程题：CPU reference

使用两个旋转 pair 构造最小 reference，并验证 position=0 时输出与输入完全相同。


In [ ]:
import numpy as np

x_even = np.array([0.25, -0.50], dtype=np.float32)
x_odd = np.array([0.75, 0.125], dtype=np.float32)
cos = np.ones_like(x_even)
sin = np.zeros_like(x_even)
out_even = x_even * cos - x_odd * sin
out_odd = x_even * sin + x_odd * cos
np.testing.assert_array_equal(out_even, x_even)
np.testing.assert_array_equal(out_odd, x_odd)
print("POSITION_ZERO_REFERENCE=PASS")


## 三、中等编程题：真实 SIMD 运行

完成 1.2 的 Host 编译后，以 2 次预热、10 次重复运行当前源码。必须取得 `status=PASS`、`cases=4/4`、`fallback=0`、`path=ASCENDC_SIMD_RTC` 和 `device_id=0`；程序非零退出或字段缺失都不能通过。


In [ ]:
%%bash
set -euo pipefail
CANN_PREFIX="${ASCEND_HOME:-${ASCEND_HOME_PATH:-${ASCEND_TOOLKIT_HOME:?请设置 ASCEND_HOME}}}"
source "$CANN_PREFIX/set_env.sh"

RESULT="$(LD_LIBRARY_PATH="$CANN_PREFIX/lib64:${LD_LIBRARY_PATH:-}" \
  ./build/rope_simd_rtc --kernel src/rope_simd_kernel.cpp --warmup 2 --repeat 10)"
echo "$RESULT"
[[ "$RESULT" == ROPE_RESULT\ status=PASS* ]]
for field in "cases=4/4" "fallback=0" "path=ASCENDC_SIMD_RTC" "device_id=0"; do
  [[ "$RESULT" == *"$field"* ]] || { echo "missing field: $field" >&2; exit 1; }
done
echo "REPORT_STATUS=PASS"


## 四、困难编程题：SIMD 参数实验

在不改变 6 个缓冲区、pair-planar 数学合同和 4 个 position case 的前提下，调整一个 tile/vector 参数：

1. 修改前后都先通过 4/4 正确性回归；
2. 固定 warmup/repeat、输入、Device 和计时口径；
3. 记录两次 `ROPE_RESULT`，比较 `device_mean_us`；
4. 只有具备同口径结果时才写更快/更慢；没有目标设备或 profiler 时结论停在 `DEFERRED`。

不要改动 `rope_simt_950.asc` 来完成本题，也不要把 Host launch+sync 时间误写成 profiler 纯 Kernel 时间。


In [ ]:
def parse_rope_result(line):
    parts = line.strip().split()
    if not parts or parts[0] != "ROPE_RESULT":
        raise ValueError("不是 ROPE_RESULT 机器行")
    fields = dict(item.split("=", 1) for item in parts[1:] if "=" in item)
    required = {"status", "cases", "max_error", "tolerance", "device_mean_us", "fallback", "path", "device_id"}
    missing = required - fields.keys()
    if missing:
        raise ValueError(f"缺少字段：{sorted(missing)}")
    assert fields["status"] == "PASS" and fields["cases"] == "4/4"
    assert float(fields["max_error"]) <= float(fields["tolerance"])
    assert fields["fallback"] == "0" and fields["path"] == "ASCENDC_SIMD_RTC"
    return fields


def compare_one_variable(baseline_line, candidate_line):
    baseline = parse_rope_result(baseline_line)
    candidate = parse_rope_result(candidate_line)
    assert baseline["device_id"] == candidate["device_id"] == "0"
    baseline_us = float(baseline["device_mean_us"])
    candidate_us = float(candidate["device_mean_us"])
    return {
        "baseline_us": baseline_us,
        "candidate_us": candidate_us,
        "delta_percent": (candidate_us - baseline_us) / baseline_us * 100.0,
        "timing_scope": "host launch + stream synchronize",
    }

# 填入两次真实 ROPE_RESULT 后再执行：
# comparison = compare_one_variable(BASELINE_RESULT, CANDIDATE_RESULT)


## 完成标准

客观题与 position=0 reference 正确；中等题输出 `REPORT_STATUS=PASS` 且字段完整；困难题先保证数学合同和 4/4 case 不变，再给出可复核的单变量比较。SIMT 状态只能写“模板已提供，待 950 实机验证”。


In [ ]:
# 完成四类考核后按需执行；Notebook 不会自动展开答案。
!cat answer/02.03_answer.md
